# Lekcia 08 - Návrhový vzor viacagentového systému


## Nastavenie


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Prečo viacagentové systémy?

Úlohy zo skutočného sveta, ako je plánovanie výletu, zahŕňajú rôzne druhy odborných znalostí — logistiku, miestne znalosti, rozpočtovanie a ďalšie. Jeden agent, ktorý sa snaží zvládnuť všetko, sa rýchlo stáva neprehľadným.

Viacagentové systémy riešia tento problém cez **špecializáciu**: každý agent sa zameriava na jednu oblasť odbornosti a dosahuje vyššiu kvalitu výsledkov než generalista. Tiež zlepšujú **škálovateľnosť** — môžete pridať nových agentov (napr. špecialistu na lety, kritika reštaurácií) bez nutnosti prepisovať existujúci pracovný tok. Agentia spolupracujú cez štruktúrovaný proces, pričom si odovzdávajú kontext od jedného k druhému.


## Tvorba špecializovaných agentov


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Vytváranie sekvenčného pracovného postupu

`WorkflowBuilder` vám umožňuje prepojiť agentov do usmerneného grafu. Tu vytvárame jednoduchý dvojstupňový proces: **TravelPlanner** vytvára itinerár a potom ho **TravelConcierge** prezerá a vylepšuje.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Pridanie ďalších agentov do pracovného postupu

Jednou z najväčších výhod vzoru s viacerými agentmi je, aké je jednoduché ho rozšíriť. Nižšie pridávame agenta **BudgetReviewer**, ktorý kontroluje plán vzhľadom na rozpočet cestujúceho, označuje položky, ktoré by mohli prekročiť limit nákladov, a navrhuje úsporné alternatívy. Pracovný postup teraz spúšťa troch agentov za sebou:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Zhrnutie

V tejto lekcii ste sa naučili ako:

1. **Vytvoriť špecializovaných agentov** — každý s konkrétnou úlohou (plánovanie, concierge, kontrola rozpočtu).
2. **Prepojiť agentov do postupného pracovného postupu** pomocou `WorkflowBuilder` a `add_edge`.
3. **Streamovať výstup** z viacagentového potrubia, sledovať, ktorý agent práve hovorí.
4. **Rozšíriť pracovný postup** pridaním nových agentov do reťazca bez úpravy existujúcich.

Dizajnový vzor viacagentového systému udržiava každého agenta jednoduchého, zatiaľ čo produkuje bohatšie, dôkladnejšie preskúmané výsledky, ako by dokázal akýkoľvek jednotlivý agent sám.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zrieknutie sa zodpovednosti**:  
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Aj keď sa snažíme o presnosť, prosím, majte na pamäti, že automatizované preklady môžu obsahovať chyby alebo nepresnosti. Originálny dokument v jeho pôvodnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie odporúčame profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne výklady vzniknuté použitím tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
